# Diffusion-LM Playground

Interactively sample from the trained masked-diffusion **DiT** (`runs/dit-v5.1/final.pt`).

This notebook lets you:
1. **Generate** TinyStories unconditionally
2. **Watch the reverse diffusion** fill an all-`[MASK]` sequence into a story
3. **Infill** — fix a prefix/suffix and diffuse the middle (the payoff of bidirectional, any-order generation)
4. An interactive **slider playground**

Tips: sample with `steps ≈ seq_len` (one token per step) and `temperature ≈ 0.9`. Too few steps or low temperature → word-salad / repetition loops.

## Setup — load the model, tokenizer, and scheduler

In [ ]:
import torch
from pathlib import Path

from diffusionlm_from_scratch.model import DiT, DiTConfig
from diffusionlm_from_scratch.scheduler import AbsorbingScheduler
from diffusionlm_from_scratch.dataset import build_tokenizer
from diffusionlm_from_scratch.trainer import sample

# Resolve the repo root so the notebook works from anywhere.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Canonical model: 142M, trained on all 2.7M TinyStories (CE 2.18). Falls back to
# the smaller v5.1 (66M, 600k stories) if the full model isn't present.
CKPT = ROOT / 'runs' / 'dit-full-cont' / 'final.pt'
if not CKPT.exists():
    CKPT = ROOT / 'runs' / 'dit-v5.1' / 'final.pt'
TOKENIZER = ROOT / 'tinystories_tokenizer'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

ckpt = torch.load(CKPT, map_location=DEVICE)
model = DiT(DiTConfig(**ckpt['config'])).to(DEVICE).eval()
model.load_state_dict(ckpt['model'])           # 'model' = EMA weights; 'raw' is also stored

tok, MASK_ID, VOCAB = build_tokenizer(str(TOKENIZER))
sched = AbsorbingScheduler(mask_id=MASK_ID)
SEQ = ckpt['config']['max_seq_len']

print(f'loaded {CKPT.parent.name}/{CKPT.name} | {sum(p.numel() for p in model.parameters())/1e6:.0f}M params '
      f'| vocab {VOCAB} | seq_len {SEQ} | device {DEVICE}')

## 1. Generate stories

`sample()` starts from an all-`[MASK]` sequence and walks the noise level down to 0, committing the most-confident tokens at each step (confidence-based parallel sampling).

In [ ]:
torch.manual_seed(0)
for i, s in enumerate(sample(model, sched, tok, seq_len=SEQ, n=4, steps=SEQ, temperature=0.9)):
    print(f'[{i}] {s}\n')

## 2. Watch the reverse diffusion

Generation is **not** inherently left-to-right — every position is predicted from the full bidirectional context. But **which** positions get committed depends on the selection rule:

- **`order='confidence'`** (default) commits the *surest* positions first. On this model, confidence is concentrated at the predictable story **opening** (max-prob ≈ 0.2–0.6 at the first tokens, but ≈ 0.08 — basically flat — further out). So it anchors on the start and **cascades front-to-back**, which looks almost autoregressive even though the mechanism isn't.
- **`order='random'`** commits random masked positions, so the sequence fills in **spread across its whole length at once** — the "any-order" parallel picture. Quality is comparable here.

`▢` marks positions still masked. The reveal count follows a **cosine (MaskGIT) schedule** (few tokens early, more later). Run the cell to compare both orders.

In [ ]:
import math

def render(seq):
    """Decode a sequence, showing still-masked positions as ▢."""
    return ''.join(' ▢' if t == MASK_ID else tok.decode([t]) for t in seq.tolist())

@torch.no_grad()
def sample_trace(steps=128, temperature=0.9, snapshots=6, order='confidence_weighted'):
    x = torch.full((1, SEQ), MASK_ID, device=DEVICE, dtype=torch.long)
    ts = torch.linspace(1, 0, steps + 1, device=DEVICE)
    snap_at = {int(i) for i in torch.linspace(0, steps - 1, snapshots).tolist()}
    snaps = []
    for i in range(steps):
        probs = (model(x, ts[i].expand(1)) / temperature).softmax(-1)[0]
        pred = torch.multinomial(probs, 1).squeeze(-1)
        conf = probs.max(-1).values
        still = x[0] == MASK_ID
        if not still.any():
            break
        r = (i + 1) / steps                                    # cosine reveal schedule
        target = (1.0 - math.cos(math.pi / 2 * r)) * SEQ
        k = min(int(still.sum()), max(1, int(round(target - int((~still).sum())))))
        conf_m = conf.masked_fill(~still, -1.0)
        if order == 'confidence':                              # surest first -> front-to-back
            idx = conf_m.topk(k).indices
        elif order == 'confidence_weighted':                   # confidence-biased but spread
            idx = torch.multinomial(conf_m.clamp(min=0.0), k, replacement=False)
        else:                                                  # random -> fully spread
            cand = still.nonzero().squeeze(-1)
            idx = cand[torch.randperm(cand.numel(), device=DEVICE)[:k]]
        x[0, idx] = pred[idx]
        if i in snap_at:
            snaps.append((i, x[0].clone()))
    snaps.append((steps, x[0].clone()))
    return snaps

# Three position-selection rules (all commit the same NUMBER per step):
#  'confidence'          -> surest positions first; cascades front-to-back.
#  'confidence_weighted' -> sample positions weighted by confidence: keeps the
#                           confidence bias (good quality) but fills SPREAD across
#                           the whole sequence, like the any-order diffusion picture.
#  'random'              -> ignore confidence; fully spread.
for order in ['confidence', 'confidence_weighted', 'random']:
    print(f'\n========== order = {order} ==========')
    torch.manual_seed(0)
    for i, seq in sample_trace(steps=256, snapshots=5, order=order):
        filled = int((seq != MASK_ID).sum())
        print(f'step {i:3d} | {filled:3d}/{SEQ} filled |{render(seq)[:150]}')

## 3. Infilling — fix a prefix/suffix, diffuse the middle

Because the model is bidirectional, you can pin some tokens and let it fill the rest, conditioning on **both** sides. Set `prefix` and/or `suffix`. (Quality is rougher than unconditional generation — the model wasn't trained specifically for infilling, and out-of-distribution prompts like "dragon" are harder than typical TinyStories.)

In [ ]:
@torch.no_grad()
def infill(prefix='', suffix='', steps=None, temperature=0.9):
    steps = steps or SEQ
    pre = tok(prefix)['input_ids'] if prefix else []
    suf = tok(suffix)['input_ids'] if suffix else []
    x = torch.full((1, SEQ), MASK_ID, device=DEVICE, dtype=torch.long)
    if pre:
        x[0, :len(pre)] = torch.tensor(pre, device=DEVICE)
    if suf:
        x[0, SEQ - len(suf):] = torch.tensor(suf, device=DEVICE)
    n_fixed = len(pre) + len(suf)
    ts = torch.linspace(1, 0, steps + 1, device=DEVICE)
    for i in range(steps):
        probs = (model(x, ts[i].expand(1)) / temperature).softmax(-1)[0]
        pred = torch.multinomial(probs, 1).squeeze(-1)
        conf = probs.max(-1).values
        still = x[0] == MASK_ID          # prefix/suffix are not masked -> never changed
        if not still.any():
            break
        # cosine reveal over the positions we're free to fill (excludes the fixed ones)
        r = (i + 1) / steps
        target = (1.0 - math.cos(math.pi / 2 * r)) * (SEQ - n_fixed) + n_fixed
        conf = conf.masked_fill(~still, -1.0)
        k = min(int(still.sum()), max(1, int(round(target - int((~still).sum())))))
        idx = conf.topk(k).indices
        x[0, idx] = pred[idx]
    return tok.decode(x[0], skip_special_tokens=True)

torch.manual_seed(0)
print('PREFIX only:\n', infill(prefix='One day, Lily and Tom went to the'), '\n')
print('PREFIX + SUFFIX:\n', infill(prefix='Tim found a', suffix='and they were very happy.'))

## 4. Interactive playground

Move the sliders, then click **Run Interact**. (Needs `ipywidgets`, which is in the dev deps.)

In [ ]:
import ipywidgets as widgets

def play(temperature, steps, n, seed, order, corrector_frac):
    torch.manual_seed(seed)
    for i, s in enumerate(sample(model, sched, tok, seq_len=SEQ, n=n, steps=steps,
                                 temperature=temperature, order=order,
                                 corrector_frac=corrector_frac)):
        print(f'[{i}] {s}\n')

# 'corrector' > 0 turns on predictor-corrector sampling: each step re-masks that
# fraction of committed tokens and re-predicts them, fixing earlier commits inline.
widgets.interact_manual(
    play,
    temperature=widgets.FloatSlider(min=0.5, max=1.2, step=0.05, value=0.9),
    steps=widgets.IntSlider(min=32, max=SEQ, step=32, value=SEQ),
    n=widgets.IntSlider(min=1, max=6, value=3),
    seed=widgets.IntSlider(min=0, max=100, value=0),
    order=widgets.Dropdown(options=['confidence', 'confidence_weighted', 'random'],
                           value='confidence'),
    corrector_frac=widgets.FloatSlider(min=0.0, max=0.3, step=0.05, value=0.0,
                                       description='corrector'),
);

## 5. Regenerate committed tokens (refinement)

The strict sampler **freezes** a token once it's committed. To let the model *fix* an earlier choice — e.g. a character name it guessed before the rest of the story existed — use `refine()`: each pass re-masks a small **random** fraction of positions and re-predicts them conditioned on the full sequence.

The fraction must be small and **random**. Re-masking *by confidence* (keeping only the surest tokens each step) collapses into repetitive high-confidence loops like `Mrs Mrs Mrs...`; gentle random remasking is stable and can correct real mistakes (e.g. an inconsistent name).

In [ ]:
# Regenerate already-committed tokens: re-mask a small RANDOM fraction each pass
# and re-predict it with the full sequence as context. The strict sampler never
# un-commits, so this is how the model fixes earlier mistakes.
from diffusionlm_from_scratch.trainer import refine

torch.manual_seed(0)
ids = sample(model, sched, tok, seq_len=SEQ, n=2, steps=SEQ, temperature=0.9, return_ids=True)
print('GENERATED:')
for s in ids:
    print(' ', tok.decode(s, skip_special_tokens=True)[:240])

ids2 = refine(model, sched, ids.clone(), passes=8, frac=0.15)
print('\nAFTER refine (8 passes, re-mask 15% each):')
for s in ids2:
    print(' ', tok.decode(s, skip_special_tokens=True)[:240])